# 0418 · Efecto sábado en validación

**Por qué.** En la prueba final, el umbral de 0,197 generó 868 alertas por día en lugar de 500, y el sábado aportó 640 de ellas (cerca del 74 %). En el documento quedaron abiertas dos preguntas (sección 5.5):

1. ¿El exceso de alertas del sábado **ya estaba en validación** (un patrón del modelo) o **apareció en julio y agosto** (deriva)?
2. ¿Cuánto se explica por el **supuesto de contar el sábado como día hábil**, que acorta el plazo de las guías que salen ese día?

**Qué NO hace.** No vuelve a correr la prueba final, no reentrena modelos y no cambia la etiqueta del modelado. Usa solo las predicciones de validación (abril a junio) que dejó `0413_RandomForest`, el umbral congelado en `P4b_decision.json` y, para comparar, la tabla agregada de la prueba que ya dejó `0420_Interpretabilidad_4` (`05_alertas_por_segmento.csv`).

| Sección | Pregunta | Salida |
| --- | --- | --- |
| A | Alertas por día de la semana en validación con el umbral congelado, frente a la prueba | `P9_a_*` |
| B | Plazo y margen en T0 del sábado | `P9_b_*` |
| C | Etiqueta con el sábado como día no hábil, en las mismas guías de validación | `P9_c_*` |
| D | Umbral por día de la semana: fijado en abril y mayo, evaluado en junio | `P9_d_*` |

**Reglas de lectura (fijadas antes de correr).**
- **A.** Si en validación el sábado ya aporta **50 % o más** de las alertas, el exceso es un patrón del modelo que ya existía. Si aporta **menos del 25 %**, hubo deriva entre validación y prueba. Entre 25 % y 50 % es mixto.
- **C.** Si con el sábado no hábil el incumplimiento de las guías que salen el sábado queda **a lo sumo 2 puntos por encima** del promedio de lunes a viernes, el efecto sábado se atribuye al supuesto del calendario. Si sigue más de 2 puntos por encima, hay algo propio del sábado.
- **D.** El umbral por día se recomienda solo si, en junio, iguala o mejora la exhaustividad del umbral único **con el mismo número de alertas o menos**.

**Salidas.** CSV en `REVISION_2026_10/V3/SABADO/`. Solo agregados: ningún número de guía, cliente ni persona.

In [0]:
%pip install "duckdb<2"

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 124.8 MB/s  0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, json, numpy as np, pandas as pd, duckdb
from sklearn.metrics import average_precision_score

PATH_DATABRICKS  = os.environ.get("PATH_DATABRICKS_0418", "/Volumes/proyectogrado/source/source/")
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/V3/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"
RUTA_GOLD_V3     = f"{CARPETA_SALIDAS}GOLD_V3_ENVIO.parquet"
RUTA_CALENDARIO  = f"{PATH_DATABRICKS}PARAMETRICAS/DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS.parquet"
RUTA_PRUEBA_SEG  = f"{CARPETA_SALIDAS}INTERPRETACION/05_alertas_por_segmento.csv"   # salida agregada de 0420_4
CARPETA_SAB      = f"{CARPETA_SALIDAS}SABADO/"
os.makedirs(CARPETA_SAB, exist_ok=True)

COL_FECHA_CAL   = "DATE"
HABIL_PRINCIPAL = "UPPER(TRIM(TIPO_DIA)) = 'DIA HABIL' AND FECHA_FESTIVA IS NULL"          # igual que 0330 y 0340
SIN_SABADO      = HABIL_PRINCIPAL + f' AND dayofweek(CAST("{COL_FECHA_CAL}" AS DATE)) <> 6'  # DuckDB: 6 = sábado
CORTE_OBSERVACION = "2026-08-31 23:59:59"
ALERTAS_POR_DIA   = 500                       # mismo escenario de 0416
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]   # pandas: 0 = lunes

try:
    display
except NameError:
    display = print

def guardar(nombre, df, index=False):
    df.to_csv(f"{CARPETA_SAB}P9_{nombre}.csv", index=index)
    display(df)
    return df

In [0]:
# Decisión congelada en 0416 (solo se lee)
DEC = json.load(open(f"{CARPETA_SALIDAS}P4b_decision.json"))
UMBRAL, MODELO_CLF = DEC["umbral"], DEC["modelo_clf"]
print(f"Modelo de clasificación: {MODELO_CLF} · umbral congelado: {UMBRAL:.4f}")

# Predicciones de validación del modelo elegido (versión 'internas'), abril a junio
pv = pd.read_parquet(f"{CARPETA_MODELADO}pred_val_{MODELO_CLF}.parquet")
pv = pv[(pv.version == "internas") & (pv.tarea == "clf")].copy()
pv["dia"] = pd.to_datetime(pv["dia"])
pv["dia_semana"] = pv.dia.dt.dayofweek
pv["alerta"] = pv.pred >= UMBRAL
print(f"{len(pv):,} guías de validación · {pv.dia.nunique()} días · bloques {sorted(pv.bloque.astype(str).unique())}")

Modelo de clasificación: RandomForest · umbral congelado: 0.1967
587,957 guías de validación · 72 días · bloques ['2026-04-01', '2026-05-01', '2026-06-01']


## A. Alertas por día de la semana en validación

Mismo cálculo que 0420_4 hizo sobre la prueba: `alertas_por_dia` es el total de alertas de ese día de la semana dividido entre **todos** los días del periodo, de modo que la columna suma el promedio diario. Se agrega `alertas_por_dia_propio`: alertas divididas entre el número de días de ese día de la semana (por ejemplo, alertas en un sábado típico).

In [0]:
def por_dia_semana(c, periodo):
    n_dias = c.dia.nunique()
    filas = []
    for ds, g in c.groupby("dia_semana"):
        filas.append({"periodo": periodo, "dia_semana": DIAS[int(ds)], "guias": len(g), "dias_de_ese_tipo": g.dia.nunique(),
                      "tasa_real": g.y.mean(), "alertas": int(g.alerta.sum()),
                      "alertas_por_dia": g.alerta.sum() / n_dias,
                      "alertas_por_dia_propio": g.alerta.sum() / g.dia.nunique(),
                      "precision": g.loc[g.alerta, "y"].mean() if g.alerta.any() else np.nan,
                      "exhaustividad": g.loc[g.y == 1, "alerta"].mean() if (g.y == 1).any() else np.nan})
    t = pd.DataFrame(filas)
    t["pct_de_las_alertas"] = 100 * t.alertas / t.alertas.sum()
    t["pct_de_las_guias"] = 100 * t.guias / t.guias.sum()
    return t

a_val = por_dia_semana(pv, "validación abr–jun")
a_bloques = pd.concat([por_dia_semana(g, f"validación {b}") for b, g in pv.groupby(pv.bloque.astype(str))], ignore_index=True)
guardar("a_validacion_dia_semana", pd.concat([a_val, a_bloques], ignore_index=True).round(4))

periodo,dia_semana,guias,dias_de_ese_tipo,tasa_real,alertas,alertas_por_dia,alertas_por_dia_propio,precision,exhaustividad,pct_de_las_alertas,pct_de_las_guias
validación abr–jun,Lunes,45838,10,0.0522,1212,16.8333,121.2,0.4274,0.2166,3.3617,7.7961
validación abr–jun,Martes,116620,13,0.0599,4340,60.2778,333.8462,0.4664,0.2896,12.0378,19.8348
validación abr–jun,Miércoles,123960,13,0.0639,4197,58.2917,322.8462,0.4539,0.2406,11.6412,21.0832
validación abr–jun,Jueves,113321,12,0.0611,4200,58.3333,350.0,0.4755,0.2885,11.6495,19.2737
validación abr–jun,Viernes,104207,11,0.0637,3008,41.7778,273.4545,0.4488,0.2033,8.3433,17.7236
validación abr–jun,Sábado,84011,13,0.1291,19096,265.2222,1468.9231,0.3245,0.5713,52.9665,14.2886
validación 2026-04-01,Lunes,20623,4,0.0536,632,26.3333,158.0,0.4826,0.276,4.2694,10.4691
validación 2026-04-01,Martes,44347,4,0.0563,1495,62.2917,373.75,0.5525,0.3307,10.0993,22.5123
validación 2026-04-01,Miércoles,44068,5,0.0633,1549,64.5417,309.8,0.4603,0.2556,10.4641,22.3707
validación 2026-04-01,Jueves,38221,4,0.0591,1270,52.9167,317.5,0.5315,0.2988,8.5793,19.4025


,periodo,dia_semana,guias,dias_de_ese_tipo,tasa_real,alertas,alertas_por_dia,alertas_por_dia_propio,precision,exhaustividad,pct_de_las_alertas,pct_de_las_guias
0,validación abr–jun,Lunes,45838,10,0.0522,1212,16.8333,121.2000,0.4274,0.2166,3.3617,7.7961
1,validación abr–jun,Martes,116620,13,0.0599,4340,60.2778,333.8462,0.4664,0.2896,12.0378,19.8348
2,validación abr–jun,Miércoles,123960,13,0.0639,4197,58.2917,322.8462,0.4539,0.2406,11.6412,21.0832
3,validación abr–jun,Jueves,113321,12,0.0611,4200,58.3333,350.0000,0.4755,0.2885,11.6495,19.2737
4,validación abr–jun,Viernes,104207,11,0.0637,3008,41.7778,273.4545,0.4488,0.2033,8.3433,17.7236
5,validación abr–jun,Sábado,84011,13,0.1291,19096,265.2222,1468.9231,0.3245,0.5713,52.9665,14.2886
6,validación 2026-04-01,Lunes,20623,4,0.0536,632,26.3333,158.0000,0.4826,0.2760,4.2694,10.4691
7,validación 2026-04-01,Martes,44347,4,0.0563,1495,62.2917,373.7500,0.5525,0.3307,10.0993,22.5123
8,validación 2026-04-01,Miércoles,44068,5,0.0633,1549,64.5417,309.8000,0.4603,0.2556,10.4641,22.3707
9,validación 2026-04-01,Jueves,38221,4,0.0591,1270,52.9167,317.5000,0.5315,0.2988,8.5793,19.4025


In [0]:
# Comparación con la prueba final (tabla agregada que ya dejó 0420_4; no se recalcula nada de la prueba)
if os.path.exists(RUTA_PRUEBA_SEG):
    pr = pd.read_csv(RUTA_PRUEBA_SEG)
    pr = pr[pr.segmento == "dia_semana"].copy()
    pr["dia_semana"] = pr.valor.astype(int).map(dict(enumerate(DIAS)))
    pr["pct_de_las_alertas"] = 100 * pr.alertas_por_dia / pr.alertas_por_dia.sum()
    comp = a_val[["dia_semana", "tasa_real", "alertas_por_dia", "pct_de_las_alertas", "precision", "exhaustividad"]].merge(
        pr[["dia_semana", "tasa_real", "alertas_por_dia", "pct_de_las_alertas", "precision_alertas", "exhaustividad"]],
        on="dia_semana", how="outer", suffixes=("_validacion", "_prueba"))
    comp = comp.sort_values("dia_semana", key=lambda s: s.map({d: i for i, d in enumerate(DIAS)}))
    guardar("a_validacion_vs_prueba", comp.round(4))
else:
    print("No está 05_alertas_por_segmento.csv de 0420_4: se omite la comparación con la prueba.")

pct_sab_val = float(a_val.loc[a_val.dia_semana == "Sábado", "pct_de_las_alertas"].sum())
lectura_a = ("patrón que ya existía en validación" if pct_sab_val >= 50 else
             "deriva entre validación y prueba" if pct_sab_val < 25 else "mixto")
print(f"El sábado aporta el {pct_sab_val:.1f} % de las alertas de validación → {lectura_a}")

dia_semana,tasa_real_validacion,alertas_por_dia_validacion,pct_de_las_alertas_validacion,precision,exhaustividad_validacion,tasa_real_prueba,alertas_por_dia_prueba,pct_de_las_alertas_prueba,precision_alertas,exhaustividad_prueba
Lunes,0.0522,16.8333,3.3617,0.4274,0.2166,0.0552,8.26,0.9512,0.3705,0.0843
Martes,0.0599,60.2778,12.0378,0.4664,0.2896,0.0496,52.0,5.9879,0.1827,0.1384
Miércoles,0.0639,58.2917,11.6412,0.4539,0.2406,0.0554,63.78,7.3444,0.2427,0.1488
Jueves,0.0611,58.3333,11.6495,0.4755,0.2885,0.0557,48.3,5.5618,0.2766,0.1411
Viernes,0.0637,41.7778,8.3433,0.4488,0.2033,0.0493,55.96,6.4439,0.2316,0.179
Sábado,0.1291,265.2222,52.9665,0.3245,0.5713,0.127,640.12,73.7109,0.1904,0.8084
Domingo,null,null,null,null,null,0.0049,0.0,0.0,null,0.0


El sábado aporta el 53.0 % de las alertas de validación → patrón que ya existía en validación


## B. Plazo y margen en T0 según el día de la semana

Si el sábado cuenta como hábil, una guía admitida el viernes con plazo de un día vence el sábado a medianoche. Aquí se mira, en las guías de validación, el plazo en días, el margen en T0 y qué parte sale con poco margen.

In [0]:
G   = f"read_parquet('{RUTA_GOLD_V3}')"
CAL = f"read_parquet('{RUTA_CALENDARIO}')"
con = duckdb.connect()
con.register("val", pv[["guia", "bloque", "dia_semana", "y", "pred", "alerta"]].assign(bloque=pv.bloque.astype(str)))

b = con.execute(f"""
    SELECT V.dia_semana, COUNT(*) AS guias,
           AVG(G.sla_dias) AS plazo_medio_dias,
           AVG((G.sla_dias = 1)::INT) AS pct_plazo_1_dia,
           MEDIAN(G.margen_sla_en_t0_h) AS margen_mediano_h,
           AVG((G.margen_sla_en_t0_h < 12)::INT) AS pct_margen_menor_12h,
           AVG((dayofweek(CAST(G.vencimiento_sla AS DATE)) = 6)::INT) AS pct_vence_en_sabado
    FROM val V JOIN {G} G USING (guia)
    GROUP BY 1 ORDER BY 1""").df()
b["dia_semana"] = b.dia_semana.astype(int).map(dict(enumerate(DIAS)))
guardar("b_plazo_margen_dia_semana", b.round(4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dia_semana,guias,plazo_medio_dias,pct_plazo_1_dia,margen_mediano_h,pct_margen_menor_12h,pct_vence_en_sabado
Lunes,45838,1.4502,0.6442,15.8989,0.002,8.0E-4
Martes,116620,1.191,0.8529,14.76,0.0154,8.0E-4
Miércoles,123960,1.1883,0.841,14.6162,0.0099,0.0116
Jueves,113321,1.2026,0.8345,14.6158,0.0076,0.0154
Viernes,104207,1.2021,0.8406,14.6767,0.0071,0.1164
Sábado,84011,1.158,0.878,15.5436,0.0016,0.8124


,dia_semana,guias,plazo_medio_dias,pct_plazo_1_dia,margen_mediano_h,pct_margen_menor_12h,pct_vence_en_sabado
0,Lunes,45838,1.4502,0.6442,15.8989,0.0020,0.0008
1,Martes,116620,1.1910,0.8529,14.7600,0.0154,0.0008
2,Miércoles,123960,1.1883,0.8410,14.6162,0.0099,0.0116
3,Jueves,113321,1.2026,0.8345,14.6158,0.0076,0.0154
4,Viernes,104207,1.2021,0.8406,14.6767,0.0071,0.1164
5,Sábado,84011,1.1580,0.8780,15.5436,0.0016,0.8124


## C. Etiqueta con el sábado como día no hábil (mismas guías de validación)

Se recalcula el vencimiento con la misma regla de 0340 (fin del n-ésimo día hábil desde la admisión, sin contar el día de admisión), cambiando solo el calendario: el sábado deja de ser hábil. Primero se reproduce la regla principal como control; debe coincidir con la etiqueta del modelado en casi todas las guías.

Después se compara, por día de la semana de la salida, el incumplimiento con las dos reglas y cómo se ven las alertas del modelo congelado frente a la etiqueta alternativa. El modelo **no** se reentrena: la pregunta es cuánto del «exceso» del sábado desaparece solo con cambiar el calendario.

In [0]:
def limite_variante(condicion_habil):
    # Límite (inicio del día siguiente al vencimiento) por guía de validación con la regla de 0340, para un calendario dado.
    return con.execute(f"""
    WITH C AS (SELECT CAST("{COL_FECHA_CAL}" AS DATE) AS fecha, bool_and(COALESCE({condicion_habil}, FALSE)) AS es_habil
               FROM {CAL} GROUP BY 1),
    HAB AS (SELECT fecha, es_habil, SUM(es_habil::INT) OVER (ORDER BY fecha) AS n_hab FROM C),
    HABIL AS (SELECT fecha, n_hab FROM HAB WHERE es_habil),
    POB AS (SELECT V.guia, G.FECHA_ENVIO AS ini, G.fecha_entrega, G.t0, CAST(G.sla_dias AS INT) AS d
            FROM val V JOIN {G} G USING (guia))
    SELECT P.guia, P.t0, P.fecha_entrega, CAST(HV.fecha AS TIMESTAMP) + INTERVAL 1 DAY AS lim
    FROM POB P
    LEFT JOIN HAB H0 ON H0.fecha = CAST(P.ini AS DATE)
    LEFT JOIN HABIL HV ON HV.n_hab = H0.n_hab + GREATEST(P.d, 0)
                          + CASE WHEN P.d <= 0 AND NOT H0.es_habil THEN 1 ELSE 0 END""").df()

def etiqueta(t):
    lim, fe = pd.to_datetime(t.lim), pd.to_datetime(t.fecha_entrega)
    y = np.where(lim.isna(), np.nan,
                 np.where(fe.notna(), (fe >= lim).astype(float),
                          np.where(lim <= pd.Timestamp(CORTE_OBSERVACION), 1.0, np.nan)))
    return pd.Series(y, index=t.index)

def horas(a, b):
    return (pd.to_datetime(a) - pd.to_datetime(b)) / pd.Timedelta(hours=1)

base = limite_variante(HABIL_PRINCIPAL).rename(columns={"lim": "lim_principal"})
alt  = limite_variante(SIN_SABADO)[["guia", "lim"]].rename(columns={"lim": "lim_sin_sabado"})
c = pv[["guia", "dia", "dia_semana", "y", "pred", "alerta"]].merge(base, on="guia").merge(alt, on="guia")
c["y_principal"]   = etiqueta(c.rename(columns={"lim_principal": "lim"}))
c["y_sin_sabado"]  = etiqueta(c.rename(columns={"lim_sin_sabado": "lim"}))
c["margen_sin_sabado_h"] = horas(c.lim_sin_sabado, c.t0)

control = pd.DataFrame([{"guias_validacion": len(pv), "guias_unidas": len(c),
                         "pct_coincide_regla_principal": 100 * (c.y_principal == c.y).mean(),
                         "pct_sin_vencimiento_sin_sabado": 100 * c.lim_sin_sabado.isna().mean()}])
guardar("c_control", control.round(3))
assert control.pct_coincide_regla_principal.iloc[0] > 99, "La regla reproducida no coincide con la etiqueta del modelado: revisar antes de leer C"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

guias_validacion,guias_unidas,pct_coincide_regla_principal,pct_sin_vencimiento_sin_sabado
587957,587957,100.0,0.0


In [0]:
cc = c.dropna(subset=["y_sin_sabado"])
filas = []
for ds, g in cc.groupby("dia_semana"):
    al = g.alerta
    filas.append({"dia_semana": DIAS[int(ds)], "guias": len(g),
                  "incumplimiento_principal": g.y.mean(), "incumplimiento_sin_sabado": g.y_sin_sabado.mean(),
                  "guias_que_dejan_de_incumplir": int(((g.y == 1) & (g.y_sin_sabado == 0)).sum()),
                  "margen_mediano_principal_h": horas(g.lim_principal, g.t0).median(),
                  "margen_mediano_sin_sabado_h": g.margen_sin_sabado_h.median(),
                  "alertas": int(al.sum()),
                  "precision_principal": g.loc[al, "y"].mean() if al.any() else np.nan,
                  "precision_sin_sabado": g.loc[al, "y_sin_sabado"].mean() if al.any() else np.nan,
                  "AP_principal": average_precision_score(g.y, g.pred) if g.y.nunique() == 2 else np.nan,
                  "AP_sin_sabado": average_precision_score(g.y_sin_sabado, g.pred) if g.y_sin_sabado.nunique() == 2 else np.nan})
c_tab = pd.DataFrame(filas)
guardar("c_etiqueta_sin_sabado_dia_semana", c_tab.round(4))

lab = c_tab[c_tab.dia_semana.isin(DIAS[:5])]
prom_lv = np.average(lab.incumplimiento_sin_sabado, weights=lab.guias)
sab = c_tab.loc[c_tab.dia_semana == "Sábado", "incumplimiento_sin_sabado"]
prom_lv_principal = np.average(lab.incumplimiento_principal, weights=lab.guias)
sab_principal = c_tab.loc[c_tab.dia_semana == "Sábado", "incumplimiento_principal"]
dif_principal = 100 * (float(sab_principal.iloc[0]) - prom_lv_principal) if len(sab_principal) else np.nan
dif = 100 * (float(sab.iloc[0]) - prom_lv) if len(sab) else np.nan
lectura_c = ("el efecto sábado se explica por el supuesto del calendario" if dif <= 2 else
             "queda un efecto propio del sábado")
resumen_c = pd.DataFrame([{"incumplimiento_global_principal": cc.y.mean(), "incumplimiento_global_sin_sabado": cc.y_sin_sabado.mean(),
                           "sabado_sin_sabado": float(sab.iloc[0]) if len(sab) else np.nan, "lunes_a_viernes_sin_sabado": prom_lv,
                           "diferencia_puntos_regla_principal": dif_principal,
                           "diferencia_puntos_sin_sabado": dif, "lectura": lectura_c}])
guardar("c_resumen", resumen_c.round(4))
print(f"Sábado frente a lunes–viernes: {dif_principal:+.1f} puntos con la regla principal; {dif:+.1f} con el sábado no hábil → {lectura_c}")

dia_semana,guias,incumplimiento_principal,incumplimiento_sin_sabado,guias_que_dejan_de_incumplir,margen_mediano_principal_h,margen_mediano_sin_sabado_h,alertas,precision_principal,precision_sin_sabado,AP_principal,AP_sin_sabado
Lunes,45838,0.0522,0.0433,408,15.8989,19.1958,1212,0.4274,0.3449,0.2517,0.1891
Martes,116620,0.0599,0.0581,212,14.76,14.8964,4340,0.4664,0.4528,0.337,0.3275
Miércoles,123960,0.0639,0.0633,68,14.6162,14.6211,4197,0.4539,0.4522,0.3023,0.3026
Jueves,113321,0.0611,0.0609,17,14.6158,14.6161,4200,0.4755,0.475,0.3519,0.3522
Viernes,104207,0.0637,0.0615,234,14.6767,14.6783,3008,0.4488,0.4432,0.2674,0.2696
Sábado,84011,0.1291,0.025,8745,15.5436,63.7392,19096,0.3245,0.0521,0.447,0.1138


incumplimiento_global_principal,incumplimiento_global_sin_sabado,sabado_sin_sabado,lunes_a_viernes_sin_sabado,diferencia_puntos_regla_principal,diferencia_puntos_sin_sabado,lectura
0.0709,0.0545,0.025,0.0594,6.7887,-3.4343,el efecto sábado se explica por el supuesto del calendario


Sábado frente a lunes–viernes: +6.8 puntos con la regla principal; -3.4 con el sábado no hábil → el efecto sábado se explica por el supuesto del calendario


## D. Umbral por día de la semana (solo validación)

Alternativa para el documento (sección 6.2): en lugar de un umbral único, uno por día de la semana, repartiendo las mismas 500 alertas diarias según el volumen de guías de cada día. Para no evaluar con los mismos datos con que se fija, los umbrales se fijan con **abril y mayo** y se evalúan en **junio**, frente al umbral único fijado de la misma forma.

In [0]:
pv["bloque_s"] = pv.bloque.astype(str)
bloques = sorted(pv.bloque_s.unique())
fij, ev = pv[pv.bloque_s.isin(bloques[:-1])], pv[pv.bloque_s == bloques[-1]]

def umbral_top_k(p, k):
    # Umbral que deja pasar las k predicciones más altas.
    k = int(round(k))
    return np.inf if k <= 0 else np.sort(np.asarray(p))[::-1][min(k, len(p)) - 1]

n_fij = fij.dia.nunique()
presupuesto = ALERTAS_POR_DIA * n_fij                                  # alertas totales permitidas en abril y mayo
u_unico = umbral_top_k(fij.pred, presupuesto)
cuota = fij.groupby("dia_semana").size() / len(fij)                     # parte de las guías de cada día
u_dia = {ds: umbral_top_k(g.pred, presupuesto * cuota[ds]) for ds, g in fij.groupby("dia_semana")}

def evaluar(e, umbrales, esquema):
    al = e.pred >= e.dia_semana.map(umbrales) if isinstance(umbrales, dict) else e.pred >= umbrales
    n = e.dia.nunique(); sab = e.dia_semana == 5
    return {"esquema": esquema, "alertas_por_dia": al.sum() / n,
            "exhaustividad": al[e.y == 1].mean(), "precision": e.y[al].mean() if al.any() else np.nan,
            "pct_alertas_en_sabado": 100 * (al & sab).sum() / max(al.sum(), 1),
            "alertas_por_sabado": (al & sab).sum() / max(e.loc[sab, "dia"].nunique(), 1)}

d_tab = pd.DataFrame([evaluar(ev, u_unico, f"umbral único ({u_unico:.3f})"),
                      evaluar(ev, u_dia, "umbral por día de la semana"),
                      evaluar(ev, UMBRAL, f"umbral congelado de 0416 ({UMBRAL:.3f}, fijado con abril a junio)")])
d_tab.insert(0, "evaluado_en", bloques[-1])
guardar("d_umbral_por_dia", d_tab.round(4))
guardar("d_umbrales_fijados", pd.DataFrame({"dia_semana": [DIAS[int(k)] for k in u_dia], "umbral": list(u_dia.values()),
                                            "cuota_de_guias": [cuota[k] for k in u_dia]}).round(4))
u, p = d_tab.iloc[0], d_tab.iloc[1]
recomienda = (p.exhaustividad >= u.exhaustividad) and (p.alertas_por_dia <= u.alertas_por_dia * 1.0001)
print("Umbral por día:", "se recomienda" if recomienda else "no mejora al umbral único con la misma carga")

evaluado_en,esquema,alertas_por_dia,exhaustividad,precision,pct_alertas_en_sabado,alertas_por_sabado
2026-06-01,umbral único (0.195),519.4783,0.3178,0.3686,42.5762,1271.75
2026-06-01,umbral por día de la semana,959.1304,0.3595,0.2258,10.5712,583.0
2026-06-01,"umbral congelado de 0416 (0.197, fijado con abril a junio)",513.0,0.3157,0.3707,42.6223,1257.25


dia_semana,umbral,cuota_de_guias
Lunes,0.1273,0.0906
Martes,0.1444,0.1988
Miércoles,0.1257,0.2111
Jueves,0.1229,0.1922
Viernes,0.1065,0.1643
Sábado,0.3774,0.1429


Umbral por día: no mejora al umbral único con la misma carga


## Cómo leer y qué entregar

- **A** responde si el sábado ya concentraba las alertas en validación. El umbral de 0416 se fijó para 500 alertas diarias **en promedio**: si el sábado ya pesaba mucho en validación, el exceso de la prueba viene sobre todo de que julio y agosto tuvieron más guías en riesgo los sábados, no de que el modelo cambiara.
- **B y C** responden cuánto pesa el supuesto del sábado hábil. Esto **no cambia** los resultados del documento: la regla principal sigue siendo la de 0330 y la prueba final no se toca. Sirve para la discusión (6.3) y para la recomendación de umbral.
- **D** da una alternativa concreta para la recomendación «umbral por día o tope diario» de la sección 6.2.

**Entregar:** la carpeta `SABADO/` (solo CSV agregados) y este cuaderno exportado en HTML.